In [1]:
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles
mp_hands = mp.solutions.hands

In [ ]:
cap = cv2.VideoCapture(0)
position=np.zeros((1,21,2))
with mp_hands.Hands(
    model_complexity=0,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as hands:
  while cap.isOpened():
    success, image = cap.read()
    if not success:
      print("Ignoring empty camera frame.")
      # If loading a video, use 'break' instead of 'continue'.
      continue

    # To improve performance, optionally mark the image as not writeable to
    # pass by reference.
    image.flags.writeable = False
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_height, image_width, _ = image.shape
    results = hands.process(image)

    # Draw the hand annotations on the image.
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    annotated_image = image.copy()
    x=[]
    y=[]

    if results.multi_hand_landmarks:  
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(
                annotated_image,
                hand_landmarks,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style())

            for point in mp_hands.HandLandmark:
                #상대좌표 (0~1)
                normalizedLandmark = hand_landmarks.landmark[point]
                x.append(normalizedLandmark.x)
                y.append(normalizedLandmark.y)
            
            #cv2.line(annotated_image, next_point[0], next_point[1], (0,0,255), 5)
    #data preprocessing
                
    # Flip the image horizontally for a selfie-view display.
  # cv2.line(image, (10,10),(350,10),(0,0,255),5)
    cv2.imshow('MediaPipe Hands', cv2.flip(annotated_image, 1))
    if cv2.waitKey(5) & 0xFF == 27:
      break
cap.release()